# SIGMOD Exp 2: Crossover Analysis

This notebook fixes a read-heavy maintained-state profile and sweeps three factors that matter for `SNAP` vs `IVMH` vs MVHT tradeoffs:

1. Delta-scan share
2. Update intensity
3. History-scan reuse share

The output is a 3-panel line figure over total latency.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pandas', 'matplotlib', 'numpy'])
print('done')

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
sys.path.append(str(ROOT / 'benches' / 'hash_join' / 'htap_simulation'))

from sigmod_exp_common import TOL, apply_paper_style, ensure_dirs, run_checked, display_name
from bench_script_functions import parse_result

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp2_crossover').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

BIN = ROOT / 'target' / 'release' / 'htap_wkld'
TABLE_TYPES = ['naive', 'ivmh', 'heap', 'chain', 'par']
BASELINES = {'naive', 'ivmh'}
REPEAT = 3
TXN_NUM = 100
TX_MAP = {'MarkTs': 'BuildSnap', 'DelSc': 'DeltaScan'}

BASE_ARGS = [
    '--txn-count', '60',
    '--warehouse-count', '10',
    '--update-ratio', '0.003',
    '--probe-ratio', '0.0001',
]
TXN_GC_RATIO = 0.05
PLOT_SERIES = [
    ('naive', ''),
    ('ivmh', ''),
    ('heap', 'Write Repair'),
    ('chain', 'Write Repair'),
    ('par', 'Write Repair'),
]
STYLE = {
    ('naive', ''): ('SNAP', TOL['red'], ':', 'x'),
    ('ivmh', ''): ('IVMH', TOL['yellow'], '--', 'P'),
    ('heap', 'Write Repair'): ('MONO-WR', TOL['blue'], '-', 'o'),
    ('chain', 'Write Repair'): ('DUAL-WR', TOL['cyan'], '-', 's'),
    ('par', 'Write Repair'): ('EPOCH-WR', TOL['green'], '-', 'D'),
}

print('ROOT   :', ROOT)
print('BIN    :', BIN)
print('OUTDIR :', DATA_DIR)

In [ ]:
print('Building htap_wkld...')
run_checked(['cargo', 'build', '--release', '--bin', 'htap_wkld'], ROOT)
print('Build OK')

In [ ]:
def merge_args(base, extra):
    merged = {}
    for args in (base, extra):
        it = iter(args)
        for token in it:
            merged[token] = next(it)
    out = []
    for k, v in merged.items():
        out.extend([k, v])
    return out


def run_single(extra_args):
    args = merge_args(BASE_ARGS, extra_args)
    rows = []
    for table_type in TABLE_TYPES:
        trials = []
        print('  table=', table_type)
        for _ in range(REPEAT):
            result = run_checked([str(BIN), *args, '--table-type', table_type], ROOT, quiet=True, timeout=180)
            df = parse_result(result.stdout, table_type)
            if df.empty:
                raise RuntimeError(f'No parsed rows for {table_type}')
            df['tx_type'] = df['tx_type'].replace(TX_MAP)
            total = df.groupby('repair_type', as_index=False)['duration_ms'].sum()
            total['table_type'] = table_type
            trials.append(total)
        df_all = pd.concat(trials, ignore_index=True)
        df_avg = df_all.groupby(['table_type', 'repair_type'], as_index=False)['duration_ms'].mean()
        if table_type in BASELINES:
            df_avg = df_avg[df_avg['repair_type'] == 'Write Repair'].copy()
            df_avg['repair_type'] = ''
        rows.append(df_avg)
    out = pd.concat(rows, ignore_index=True)
    out['total_ms'] = out['duration_ms'] / TXN_NUM
    return out


def sweep_factor(csv_name, x_col, values, make_args_fn):
    csv_path = DATA_DIR / csv_name
    if csv_path.exists():
        print('Using cached CSV:', csv_path.name)
        return pd.read_csv(csv_path, keep_default_na=False)
    all_rows = []
    for value in values:
        print(f'Running {x_col}={value}')
        df = run_single(make_args_fn(value))
        df[x_col] = value
        all_rows.append(df)
    out = pd.concat(all_rows, ignore_index=True)
    out.to_csv(csv_path, index=False)
    print('Saved', csv_path.name)
    return out


def delta_args(delta_ratio):
    analytical = 0.80
    update = 1.0 - analytical
    remaining = analytical - delta_ratio
    return [
        '--txn-update-ratio', str(update),
        '--txn-delta-ratio', str(delta_ratio),
        '--txn-probe-ratio', str(remaining / 2.0),
        '--txn-scan-ratio', str(remaining / 2.0),
        '--txn-gc-ratio', str(TXN_GC_RATIO),
    ]


def update_args(update_ratio):
    return [
        '--analytical-ratio', '0.80',
        '--txn-gc-ratio', str(TXN_GC_RATIO),
        '--update-ratio', str(update_ratio),
    ]


def reuse_args(reuse_ratio):
    return [
        '--analytical-ratio', '0.80',
        '--txn-gc-ratio', str(TXN_GC_RATIO),
        '--scan-reuse-ratio', str(reuse_ratio),
    ]

DELTA_VALUES = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25]
UPDATE_VALUES = [0.01, 0.05, 0.10, 0.15, 0.20]
REUSE_VALUES = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

df_delta = sweep_factor('sigmod_exp2_delta.csv', 'delta_ratio', DELTA_VALUES, delta_args)
df_update = sweep_factor('sigmod_exp2_update.csv', 'update_ratio', UPDATE_VALUES, update_args)
df_reuse = sweep_factor('sigmod_exp2_reuse.csv', 'reuse_ratio', REUSE_VALUES, reuse_args)

display(df_delta.head())

In [ ]:
def plot_one(ax, df, x_col, xlabel, title):
    for key in PLOT_SERIES:
        label, color, linestyle, marker = STYLE[key]
        table_type, repair_type = key
        sub = df[(df['table_type'] == table_type) & (df['repair_type'] == repair_type)]
        if sub.empty:
            continue
        ax.plot(sub[x_col], sub['total_ms'], color=color, linestyle=linestyle, marker=marker, linewidth=1.8, markersize=5, label=label)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Duration (ms / tx)')
    ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)

fig, axes = plt.subplots(1, 3, figsize=(13.6, 4.2), sharey=False)
plot_one(axes[0], df_delta, 'delta_ratio', 'Delta-Scan Share', 'Delta Sweep')
plot_one(axes[1], df_update, 'update_ratio', 'Update Intensity', 'Update Sweep')
plot_one(axes[2], df_reuse, 'reuse_ratio', 'History-Scan Reuse', 'Reuse Sweep')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=5, bbox_to_anchor=(0.5, 1.08), framealpha=0.95)
fig.tight_layout()
out_pdf = FIGS_DIR / 'sigmod_exp2_crossover.pdf'
fig.savefig(out_pdf, format='pdf')
plt.show()
print('Saved', out_pdf)